# Experimentação

Este notebook orquestra a Fase 1 e 2 da etapa de experimentação, seguindo o protocolo descrito em `docs/experimentation.md`.

## Objetivos

## Setups e Imports

In [1]:
import sys
from pathlib import Path

# Garante que a raiz do projeto está no sys.path
ROOT = Path.cwd().resolve()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "pyproject.toml").exists() and (candidate / "src").exists():
        ROOT = candidate
        break
else:
    raise RuntimeError("Could not locate project root.")

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Manipulação de Dados
import pandas as pd
import numpy as np

# Visualização de Dados
import matplotlib.pyplot as plt
import seaborn as sns

# Pré-processamento e Modelagem
from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_validate,
    GridSearchCV,
)


from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.feature_selection import SelectKBest, f_classif, mutual_info_classif
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline


# Métricas de Avaliação
from sklearn.metrics import (
    roc_auc_score,
    recall_score,
    precision_score,
    f1_score,
    make_scorer
)


# Modelos de Machine Learning
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from src.utils.exp import (
    MLPClassifierWrapper,
    build_k_grid,
    extract_selected_feature_names,
    format_selected_features_log,
    get_processed_feature_names,
    summarize_grid_search_results,
)

# importando os transformers customizados
from src.features.geo_transformer import GeoTransformer
from src.features.feature_engineer_transformer import FeatureEngineerTransformer


from sklearn.model_selection import GridSearchCV
from sklearn.feature_selection import SelectKBest, f_classif, mutual_info_classif

from src.utils.exp import (
    build_k_grid,
    extract_selected_feature_names,
    format_selected_features_log,
    get_processed_feature_names,
    summarize_grid_search_results,
)


# PyTorch
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset


# MLflow
# ATENCAO: o servidor do MLflow precisa estar rodando antes de executar este notebook.
# Em um terminal separado, execute:
#   mlflow server --host 127.0.0.1 --port 5000
# Sem isso, as celulas de tracking falharao com ConnectionRefusedError.
import hashlib
import mlflow
import mlflow.sklearn

mlflow.set_tracking_uri("http://localhost:5000")

## Processamento dos Dados

In [2]:
# dataload
df = pd.read_excel('../data/raw/Telco_customer_churn.xlsx')
df.head()

,CustomerID,Count,Country,State,City,Zip Code,Lat Long,Latitude,Longitude,Gender,...,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Label,Churn Value,Churn Score,CLTV,Churn Reason
0,3668-QPYBK,1,United States,California,Los Angeles,90003,"33.964131, -118.272783",33.964131,-118.272783,Male,...,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1,86,3239,Competitor made better offer
1,9237-HQITU,1,United States,California,Los Angeles,90005,"34.059281, -118.30742",34.059281,-118.307420,Female,...,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1,67,2701,Moved
2,9305-CDSKC,1,United States,California,Los Angeles,90006,"34.048013, -118.293953",34.048013,-118.293953,Female,...,Month-to-month,Yes,Electronic check,99.65,820.5,Yes,1,86,5372,Moved
3,7892-POOKP,1,United States,California,Los Angeles,90010,"34.062125, -118.315709",34.062125,-118.315709,Female,...,Month-to-month,Yes,Electronic check,104.80,3046.05,Yes,1,84,5003,Moved
4,0280-XJGEX,1,United States,California,Los Angeles,90015,"34.039224, -118.266293",34.039224,-118.266293,Male,...,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.3,Yes,1,89,5340,Competitor had better devices


In [3]:
# Informações gerais sobre o dataset
print("=== INFORMAÇÕES GERAIS DO DATASET ===\n")
print(df.info())

# Shape
print("\n=== SHAPE DO DATASET ===")
print(f"Linhas: {df.shape[0]}, Colunas: {df.shape[1]}")

# Colunas
print("\n=== COLUNAS DO DATASET ===")
print(df.columns.tolist())

=== INFORMAÇÕES GERAIS DO DATASET ===

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 33 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   CustomerID         7043 non-null   object 
 1   Count              7043 non-null   int64  
 2   Country            7043 non-null   object 
 3   State              7043 non-null   object 
 4   City               7043 non-null   object 
 5   Zip Code           7043 non-null   int64  
 6   Lat Long           7043 non-null   object 
 7   Latitude           7043 non-null   float64
 8   Longitude          7043 non-null   float64
 9   Gender             7043 non-null   object 
 10  Senior Citizen     7043 non-null   object 
 11  Partner            7043 non-null   object 
 12  Dependents         7043 non-null   object 
 13  Tenure Months      7043 non-null   int64  
 14  Phone Service      7043 non-null   object 
 15  Multiple Lines     7043 non-null 

In [4]:
# Transformação da coluna 'Total Charges' para numérica, tratando erros e preenchendo valores ausentes com 0.
df['Total Charges'] = pd.to_numeric(df['Total Charges'], errors='coerce')
df['Total Charges'] = df['Total Charges'].fillna(0)

In [5]:
# dropando colunas irrelevantes para a modelagem
drop_cols = [
    'Country',
    'State',
    'Lat Long',
    'Churn Label',
    'Churn Reason',
    'Count'
]


df.drop(columns=drop_cols, inplace=True)

In [6]:
target = "Churn Value"
meta_cols = ["CLTV", "CustomerID"]

feature_cols = [
    col for col in df.columns
    if col not in [target] + meta_cols
]

X = df[feature_cols]
y = df[target]

metadata = df[meta_cols]


## Splits e Validação

In [7]:
# Protocolo de validação cruzada
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


# 
X_train_val, X_test, y_train_val, y_test, metadata_train_val, metadata_test = train_test_split(
    X,
    y,
    metadata,
    test_size=0.3,
    stratify=y,
    random_state=42
)

# Checando os Splits
print("=== SPLITS ===")
print(f"Treino/Validação: {X_train_val.shape[0]} amostras")
print(f"Teste: {X_test.shape[0]} amostras")

# Checando a distribuição da variável alvo nos splits
print("\n=== DISTRIBUIÇÃO DA VARIÁVEL ALVO NOS SPLITS ===")
print("Treino/Validação:")
print(y_train_val.value_counts(normalize=True))
print("\nTeste:")
print(y_test.value_counts(normalize=True))

# Checando o formato dos dados
print("\n=== FORMATO DOS DADOS ===")
print(f"X_train_val: {X_train_val.shape}")
print(f"y_train_val: {y_train_val.shape}")
print(f"X_test: {X_test.shape}")
print(f"y_test: {y_test.shape}")


=== SPLITS ===
Treino/Validação: 4930 amostras
Teste: 2113 amostras

=== DISTRIBUIÇÃO DA VARIÁVEL ALVO NOS SPLITS ===
Treino/Validação:
Churn Value
0    0.734686
1    0.265314
Name: proportion, dtype: float64

Teste:
Churn Value
0    0.734501
1    0.265499
Name: proportion, dtype: float64

=== FORMATO DOS DADOS ===
X_train_val: (4930, 24)
y_train_val: (4930,)
X_test: (2113, 24)
y_test: (2113,)


## Baseline Inicial

- Treina Dummy e Regressão Logística e compara com a MLP e outros modelos de árvores.

In [8]:
# Definindo a etapa de pré-processamento para variáveis categóricas com OHE e numéricas com passthrough
ohe = OneHotEncoder(handle_unknown="ignore")
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", ohe, make_column_selector(dtype_include=["object", "category"])),
        ("num", "passthrough", make_column_selector(dtype_exclude=["object", "category"])),
    ],
    remainder="drop",
)

# Dicionário de Pipelines para cada modelo
baseline_params = dict(
    drop_churn_score=True,
    add_engagement_score=False,
    add_tenure_group=False,
    add_tenure_log=False,
    add_contract_ordinal=False,
    add_family_stability=False,
    add_fiber_no_support=False,
    add_support_gap_count=False,
    add_payment_automatic_flag=False,
    add_electronic_check_flag=False,
    add_paperless_echeck_flag=False,
    add_price_pressure_ratio=False,
)

models = {
    "Dummy": Pipeline([
        ("fe", FeatureEngineerTransformer(**baseline_params)),
        ("geo", GeoTransformer(
            strategy="drop",
        )),
        ("prep", preprocessor),
        ("model", DummyClassifier(strategy="most_frequent")),
    ]),
    "LogisticRegression": Pipeline([
        ("fe", FeatureEngineerTransformer(**baseline_params)),
        ("geo", GeoTransformer(
            strategy="drop",
        )),
        ("prep", preprocessor),
        ("scaler", StandardScaler(with_mean=False)),
        ("model", LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=42,
        )),
    ]),
    "DecisionTree": Pipeline([
        ("fe", FeatureEngineerTransformer(**baseline_params)),
        ("geo", GeoTransformer(
            strategy="drop",
        )),
        ("prep", preprocessor),
        ("model", DecisionTreeClassifier(
            random_state=42,
            class_weight="balanced",
        )),
    ]),
    "RandomForest": Pipeline([
        ("fe", FeatureEngineerTransformer(**baseline_params)),
        ("geo", GeoTransformer(
            strategy="drop",
        )),
        ("prep", preprocessor),
        ("model", RandomForestClassifier(
            random_state=42,
            n_jobs=-1,
            class_weight="balanced_subsample",
        )),
    ]),
    "XGBoost": Pipeline([
        ("fe", FeatureEngineerTransformer(**baseline_params)),
        ("geo", GeoTransformer(
            strategy="drop",
        )),
        ("prep", preprocessor),
        ("model", XGBClassifier(
            objective="binary:logistic",
            eval_metric="logloss",
            scale_pos_weight=(y_train_val == 0).sum() / (y_train_val == 1).sum(),
            random_state=42,
            n_jobs=-1,
        )),
    ]),
    "MLP": Pipeline([
        ("fe", FeatureEngineerTransformer(**baseline_params)),
        ("geo", GeoTransformer(
            strategy="drop",
        )),
        ("prep", preprocessor),
        ("scaler", StandardScaler(with_mean=False)),
        ("model", MLPClassifierWrapper(
            hidden_dim=64,
            batch_size=64,
            lr=1e-3,
            weight_decay=1e-5,
            max_epochs=80,
            patience=8,
            val_size=0.15,
            threshold=0.5,
            random_state=42,
            verbose=False,
        )),
    ]),
}


In [9]:
# definindo o scoring para avaliação dos modelos
scoring = {
    "pr_auc": "average_precision",
    "roc_auc": "roc_auc",
    "recall": make_scorer(recall_score, zero_division=0),
    "precision": make_scorer(precision_score, zero_division=0),
    "f1": make_scorer(f1_score, zero_division=0),
}

In [10]:
metrics = ["pr_auc", "roc_auc", "recall", "precision", "f1"]
rows = []
fold_results = {}

for model_name, estimator in models.items():
    print(f"=== AVALIANDO MODELO: {model_name} ===")
    cv_res = cross_validate(
        estimator=estimator,
        X=X_train_val,
        y=y_train_val,
        cv=cv,
        scoring=scoring,
        n_jobs=1,
        return_train_score=False,
    )

    fold_results[model_name] = pd.DataFrame({
        "fold": np.arange(1, len(cv_res["fit_time"]) + 1),
        **{m: cv_res[f"test_{m}"] for m in metrics},
        "fit_time_s": cv_res["fit_time"],
        "score_time_s": cv_res["score_time"],
    })

    rows.append({
        "model": model_name,
        **{f"{m}_mean": cv_res[f"test_{m}"].mean() for m in metrics},
        "fit_time_mean_s": cv_res["fit_time"].mean(),
        "score_time_mean_s": cv_res["score_time"].mean(),
    })

results_cv = (
    pd.DataFrame(rows)
    .sort_values("pr_auc_mean", ascending=False)
    .reset_index(drop=True)
)

print("\n=== RESULTADOS MÉDIOS DA VALIDAÇÃO CRUZADA ===")
display(results_cv.round(4))

=== AVALIANDO MODELO: Dummy ===
=== AVALIANDO MODELO: LogisticRegression ===
=== AVALIANDO MODELO: DecisionTree ===
=== AVALIANDO MODELO: RandomForest ===
=== AVALIANDO MODELO: XGBoost ===
=== AVALIANDO MODELO: MLP ===

=== RESULTADOS MÉDIOS DA VALIDAÇÃO CRUZADA ===


,model,pr_auc_mean,roc_auc_mean,recall_mean,precision_mean,f1_mean,fit_time_mean_s,score_time_mean_s
0,LogisticRegression,0.6780,0.8575,0.8142,0.5314,0.6430,0.0244,0.0113
1,MLP,0.6685,0.8556,0.8012,0.5261,0.6350,1.0713,0.0149
2,XGBoost,0.6492,0.8438,0.6743,0.5685,0.6167,0.0651,0.0157
3,RandomForest,0.6323,0.8393,0.5091,0.6467,0.5692,0.1620,0.0584
4,DecisionTree,0.3927,0.6684,0.5099,0.5147,0.5120,0.0227,0.0112
5,Dummy,0.2653,0.5000,0.0000,0.0000,0.0000,0.0105,0.0111


### Validando o Wrapper

- Aplicação da MLP fora do pipeline com o ciclo manual de validação para validar o resultado do wrapper.

In [11]:
import time
import numpy as np
import pandas as pd
import scipy.sparse as sp
import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.base import clone
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    recall_score,
    precision_score,
    f1_score,
)

from src.models.mlp import MLP, evaluate, train_with_early_stopping

# ---------- Config ----------
MLP_EPOCHS = 80
MLP_BATCH_SIZE = 64
MLP_LR = 1e-3
MLP_WD = 1e-5
MLP_HIDDEN_DIM = 64
MLP_THRESHOLD = 0.5

ES_PATIENCE = 8
ES_VAL_SIZE = 0.15

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def _rows(X, idx):
    return X.iloc[idx] if hasattr(X, "iloc") else X[idx]


def _to_dense_float32(x):
    if sp.issparse(x):
        x = x.toarray()
    return np.asarray(x, dtype=np.float32)


baseline_fe = FeatureEngineerTransformer(**baseline_params)
baseline_geo = GeoTransformer(strategy="drop")

mlp_manual_folds = []

for fold, (tr_idx, va_idx) in enumerate(cv.split(X_train_val, y_train_val), start=1):
    fit_start = time.perf_counter()

    X_tr_raw = _rows(X_train_val, tr_idx)
    X_va_raw = _rows(X_train_val, va_idx)
    y_tr = np.asarray(_rows(y_train_val, tr_idx), dtype=np.float32)
    y_va = np.asarray(_rows(y_train_val, va_idx), dtype=np.float32)

    X_tr_base = baseline_fe.fit_transform(X_tr_raw, y_tr)
    X_tr_base = baseline_geo.fit_transform(X_tr_base, y_tr)
    X_va_base = baseline_fe.transform(X_va_raw)
    X_va_base = baseline_geo.transform(X_va_base)

    prep_fold = clone(preprocessor)
    X_tr_enc = prep_fold.fit_transform(X_tr_base, y_tr)
    X_va_enc = prep_fold.transform(X_va_base)

    idx_all = np.arange(len(y_tr))
    idx_tr, idx_es = train_test_split(
        idx_all,
        test_size=ES_VAL_SIZE,
        stratify=y_tr,
        random_state=42 + fold,
    )

    X_tr_fit = X_tr_enc[idx_tr]
    y_tr_fit = y_tr[idx_tr]
    X_tr_es = X_tr_enc[idx_es]
    y_tr_es = y_tr[idx_es]

    scaler = StandardScaler(with_mean=False)
    X_tr_fit = scaler.fit_transform(X_tr_fit)
    X_tr_es = scaler.transform(X_tr_es)
    X_va_sc = scaler.transform(X_va_enc)

    X_tr_fit = _to_dense_float32(X_tr_fit)
    X_tr_es = _to_dense_float32(X_tr_es)
    X_va_sc = _to_dense_float32(X_va_sc)

    train_loader = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(
            torch.tensor(X_tr_fit, dtype=torch.float32),
            torch.tensor(y_tr_fit, dtype=torch.float32),
        ),
        batch_size=MLP_BATCH_SIZE,
        shuffle=True,
    )
    es_loader = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(
            torch.tensor(X_tr_es, dtype=torch.float32),
            torch.tensor(y_tr_es, dtype=torch.float32),
        ),
        batch_size=MLP_BATCH_SIZE,
        shuffle=False,
    )

    model = MLP(
        input_dim=X_tr_fit.shape[1],
        hidden_dim=MLP_HIDDEN_DIM,
        output_dim=1,
    ).to(DEVICE)

    pos = float((y_tr_fit == 1).sum())
    neg = float((y_tr_fit == 0).sum())
    pos_weight = torch.tensor([neg / max(pos, 1.0)], dtype=torch.float32).to(DEVICE)

    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = optim.Adam(model.parameters(), lr=MLP_LR, weight_decay=MLP_WD)

    epochs_trained = train_with_early_stopping(
        model,
        train_loader,
        es_loader,
        optimizer,
        criterion,
        device=DEVICE,
        max_epochs=MLP_EPOCHS,
        patience=ES_PATIENCE,
        threshold=MLP_THRESHOLD,
    )
    best_es_loss, _ = evaluate(
        model,
        es_loader,
        criterion,
        device=DEVICE,
        threshold=MLP_THRESHOLD,
    )
    fit_time_s = time.perf_counter() - fit_start

    score_start = time.perf_counter()
    X_va_t = torch.tensor(X_va_sc, dtype=torch.float32).to(DEVICE)

    model.eval()
    with torch.no_grad():
        logits = model(X_va_t).squeeze(1)
        prob = torch.sigmoid(logits).cpu().numpy()

    pred = (prob >= MLP_THRESHOLD).astype(int)

    mlp_manual_folds.append(
        {
            "fold": fold,
            "epochs_trained": epochs_trained,
            "best_es_loss": best_es_loss,
            "pr_auc": average_precision_score(y_va, prob),
            "roc_auc": roc_auc_score(y_va, prob),
            "recall": recall_score(y_va, pred, zero_division=0),
            "precision": precision_score(y_va, pred, zero_division=0),
            "f1": f1_score(y_va, pred, zero_division=0),
            "fit_time_s": fit_time_s,
            "score_time_s": time.perf_counter() - score_start,
        }
    )

# resultados por fold
mlp_fold_results = pd.DataFrame(mlp_manual_folds)

# médias para comparar com a versão fora do pipeline
mlp_cv_summary = pd.DataFrame([
    {
        "model": "MLP_manual",
        "pr_auc_mean": mlp_fold_results["pr_auc"].mean(),
        "roc_auc_mean": mlp_fold_results["roc_auc"].mean(),
        "recall_mean": mlp_fold_results["recall"].mean(),
        "precision_mean": mlp_fold_results["precision"].mean(),
        "f1_mean": mlp_fold_results["f1"].mean(),
        "fit_time_mean_s": mlp_fold_results["fit_time_s"].mean(),
        "score_time_mean_s": mlp_fold_results["score_time_s"].mean(),
    }
])

#display(mlp_fold_results.round(4))
display(mlp_cv_summary.round(4))



,model,pr_auc_mean,roc_auc_mean,recall_mean,precision_mean,f1_mean,fit_time_mean_s,score_time_mean_s
0,MLP_manual,0.6618,0.8505,0.7966,0.5252,0.6319,0.8568,0.0036


### Conclusão

Na tabela de baselines, a `LogisticRegression` apresentou o melhor desempenho geral, com `PR-AUC = 0.6782` e `ROC-AUC = 0.8576`, ficando levemente acima da `MLP` (`PR-AUC = 0.6728` e `ROC-AUC = 0.8551`). Ainda assim, a diferença entre os dois modelos foi pequena, e a `MLP` superou o benchmark de árvore selecionado nesta rodada, o `XGBoost` (`PR-AUC = 0.6492`). Isso indica que, já na base original, a MLP se mostrou competitiva em relação ao baseline linear e ao benchmark não linear.

Na validação da implementação, a `MLP` dentro do `Pipeline` manteve desempenho consistente e até ligeiramente superior à versão manual fora do pipeline, que obteve `PR-AUC = 0.6607` e `ROC-AUC = 0.8528`. Com isso, o wrapper foi validado com sucesso para uso no fluxo de experimentação, trazendo a vantagem de encapsular pré-processamento e validação cruzada dentro da mesma estrutura, com menor risco de leakage e maior facilidade para evoluir o pipeline com feature engineering e seleção de features.


### Logging no MLflow

## Feature Engineering

**Objetivo**: 
- Adicionar poder preditivo aos modelos de forma controlada

### Round 1 - FE orientada a hipótese

- adicionando features a partir de hipóteses construídas a partir da EDA;

In [12]:
round1_fe_params = dict(
    drop_churn_score=True,
    add_engagement_score=True,
    add_tenure_group=True,
    add_tenure_log=True,
    add_contract_ordinal=True,
    add_family_stability=True,
    add_fiber_no_support=True,
    add_support_gap_count=True,
    add_payment_automatic_flag=True,
    add_electronic_check_flag=True,
    add_paperless_echeck_flag=True,
    add_price_pressure_ratio=True,
)


models = {
    "LogisticRegression": Pipeline([
        ("fe", FeatureEngineerTransformer(
            **round1_fe_params
        )),
        ("geo", GeoTransformer(
            strategy="drop",
        )),
        ("prep", preprocessor),
        ("scaler", StandardScaler(with_mean=False)),
        ("model", LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=42,
        )),
    ]),
    "XGBoost": Pipeline([
        ("fe", FeatureEngineerTransformer(
            **round1_fe_params
        )),
        ("geo", GeoTransformer(
            strategy="drop",
        )),
        ("prep", preprocessor),
        ("model", XGBClassifier(
            objective="binary:logistic",
            eval_metric="logloss",
            scale_pos_weight=(y_train_val == 0).sum() / (y_train_val == 1).sum(),
            random_state=42,
            n_jobs=-1,
        )),
    ]),
    "MLP": Pipeline([
        ("fe", FeatureEngineerTransformer(
            **round1_fe_params
        )),
        ("geo", GeoTransformer(
            strategy="drop",
        )),
        ("prep", preprocessor),
        ("scaler", StandardScaler(with_mean=False)),
        ("model", MLPClassifierWrapper(
            hidden_dim=64,
            batch_size=64,
            lr=1e-3,
            weight_decay=1e-5,
            max_epochs=80,
            patience=8,
            val_size=0.15,
            threshold=0.5,
            random_state=42,
            verbose=False,
        )),
    ]),
}

In [13]:
metrics = ["pr_auc", "roc_auc", "recall", "precision", "f1"]
rows = []
fold_results = {}

for model_name, estimator in models.items():
    print(f"=== AVALIANDO MODELO: {model_name} ===")

    cv_res = cross_validate(
        estimator=estimator,
        X=X_train_val,
        y=y_train_val,
        cv=cv,
        scoring=scoring,
        n_jobs=1,
        return_train_score=False,
    )

    fold_results[model_name] = pd.DataFrame({
        "fold": np.arange(1, len(cv_res["fit_time"]) + 1),
        **{metric: cv_res[f"test_{metric}"] for metric in metrics},
        "fit_time_s": cv_res["fit_time"],
        "score_time_s": cv_res["score_time"],
    })

    rows.append({
        "model": model_name,
        **{f"{metric}_mean": cv_res[f"test_{metric}"].mean() for metric in metrics},
        "fit_time_mean_s": cv_res["fit_time"].mean(),
        "score_time_mean_s": cv_res["score_time"].mean(),
    })

results_cv = (
    pd.DataFrame(rows)
    .sort_values("pr_auc_mean", ascending=False)
    .reset_index(drop=True)
)

print("\n=== RESULTADOS MÉDIOS DA VALIDAÇÃO CRUZADA ===")
display(results_cv.round(4))

=== AVALIANDO MODELO: LogisticRegression ===
=== AVALIANDO MODELO: XGBoost ===
=== AVALIANDO MODELO: MLP ===

=== RESULTADOS MÉDIOS DA VALIDAÇÃO CRUZADA ===


,model,pr_auc_mean,roc_auc_mean,recall_mean,precision_mean,f1_mean,fit_time_mean_s,score_time_mean_s
0,LogisticRegression,0.6893,0.8625,0.8234,0.5382,0.6508,0.0558,0.0204
1,MLP,0.6838,0.8597,0.8226,0.5318,0.6455,0.7769,0.0259
2,XGBoost,0.6465,0.8418,0.6636,0.5752,0.6160,0.0735,0.0284


### Round 2 - Adicionando `Churn Score`

- Verificando o ganho com stacking de outro modelo;

In [14]:
round2_fe_params = dict(
    drop_churn_score=False,
    add_engagement_score=True,
    add_tenure_group=True,
    add_tenure_log=True,
    add_contract_ordinal=True,
    add_family_stability=True,
    add_fiber_no_support=True,
    add_support_gap_count=True,
    add_payment_automatic_flag=True,
    add_electronic_check_flag=True,
    add_paperless_echeck_flag=True,
    add_price_pressure_ratio=True,
)


models = {
    "LogisticRegression": Pipeline([
        ("fe", FeatureEngineerTransformer(
            **round2_fe_params
        )),
        ("geo", GeoTransformer(
            strategy="drop",
        )),
        ("prep", preprocessor),
        ("scaler", StandardScaler(with_mean=False)),
        ("model", LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=42,
        )),
    ]),
    "XGBoost": Pipeline([
        ("fe", FeatureEngineerTransformer(
            **round2_fe_params
        )),
        ("geo", GeoTransformer(
            strategy="drop",
        )),
        ("prep", preprocessor),
        ("model", XGBClassifier(
            objective="binary:logistic",
            eval_metric="logloss",
            scale_pos_weight=(y_train_val == 0).sum() / (y_train_val == 1).sum(),
            random_state=42,
            n_jobs=-1,
        )),
    ]),
    "MLP": Pipeline([
        ("fe", FeatureEngineerTransformer(
            **round2_fe_params
        )),
        ("geo", GeoTransformer(
            strategy="drop",
        )),
        ("prep", preprocessor),
        ("scaler", StandardScaler(with_mean=False)),
        ("model", MLPClassifierWrapper(
            hidden_dim=64,
            batch_size=64,
            lr=1e-3,
            weight_decay=1e-5,
            max_epochs=80,
            patience=8,
            val_size=0.15,
            threshold=0.5,
            random_state=42,
            verbose=False,
        )),
    ]),
}

In [15]:
metrics = ["pr_auc", "roc_auc", "recall", "precision", "f1"]
rows = []
fold_results = {}

for model_name, estimator in models.items():
    print(f"=== AVALIANDO MODELO: {model_name} ===")

    cv_res = cross_validate(
        estimator=estimator,
        X=X_train_val,
        y=y_train_val,
        cv=cv,
        scoring=scoring,
        n_jobs=1,
        return_train_score=False,
    )

    fold_results[model_name] = pd.DataFrame({
        "fold": np.arange(1, len(cv_res["fit_time"]) + 1),
        **{metric: cv_res[f"test_{metric}"] for metric in metrics},
        "fit_time_s": cv_res["fit_time"],
        "score_time_s": cv_res["score_time"],
    })

    rows.append({
        "model": model_name,
        **{f"{metric}_mean": cv_res[f"test_{metric}"].mean() for metric in metrics},
        "fit_time_mean_s": cv_res["fit_time"].mean(),
        "score_time_mean_s": cv_res["score_time"].mean(),
    })

results_cv = (
    pd.DataFrame(rows)
    .sort_values("pr_auc_mean", ascending=False)
    .reset_index(drop=True)
)

print("\n=== RESULTADOS MÉDIOS DA VALIDAÇÃO CRUZADA ===")
display(results_cv.round(4))

=== AVALIANDO MODELO: LogisticRegression ===
=== AVALIANDO MODELO: XGBoost ===
=== AVALIANDO MODELO: MLP ===

=== RESULTADOS MÉDIOS DA VALIDAÇÃO CRUZADA ===


,model,pr_auc_mean,roc_auc_mean,recall_mean,precision_mean,f1_mean,fit_time_mean_s,score_time_mean_s
0,XGBoost,0.9512,0.9802,0.8738,0.8404,0.8566,0.0692,0.0272
1,LogisticRegression,0.9393,0.9759,0.9297,0.7827,0.8497,0.0446,0.0210
2,MLP,0.9350,0.9741,0.9258,0.7737,0.8420,1.1711,0.0239


### Round 3 - Testando encoding para City

Definindo as mesmas features para todos as estratégias de encoding

In [16]:
round3_fe_params = dict(
    drop_churn_score=True,
    add_engagement_score=True,
    add_tenure_group=True,
    add_tenure_log=True,
    add_contract_ordinal=True,
    add_family_stability=True,
    add_fiber_no_support=True,
    add_support_gap_count=True,
    add_payment_automatic_flag=True,
    add_electronic_check_flag=True,
    add_paperless_echeck_flag=True,
    add_price_pressure_ratio=True,
)


Frequency Encoding

In [17]:
encoding_strategy = "frequency"

models = {
    "LogisticRegression": Pipeline([
        ("fe", FeatureEngineerTransformer(**round3_fe_params)),
        ("geo", GeoTransformer(
            strategy=encoding_strategy,
        )),
        ("prep", preprocessor),
        ("scaler", StandardScaler(with_mean=False)),
        ("model", LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=42,
        )),
    ]),
    "XGBoost": Pipeline([
        ("fe", FeatureEngineerTransformer(**round3_fe_params)),
        ("geo", GeoTransformer(
            strategy=encoding_strategy,
        )),
        ("prep", preprocessor),
        ("model", XGBClassifier(
            objective="binary:logistic",
            eval_metric="logloss",
            scale_pos_weight=(y_train_val == 0).sum() / (y_train_val == 1).sum(),
            random_state=42,
            n_jobs=-1,
        )),
    ]),
    "MLP": Pipeline([
        ("fe", FeatureEngineerTransformer(**round3_fe_params)),
        ("geo", GeoTransformer(
            strategy=encoding_strategy,
        )),
        ("prep", preprocessor),
        ("scaler", StandardScaler(with_mean=False)),
        ("model", MLPClassifierWrapper(
            hidden_dim=64,
            batch_size=64,
            lr=1e-3,
            weight_decay=1e-5,
            max_epochs=80,
            patience=8,
            val_size=0.15,
            threshold=0.5,
            random_state=42,
            verbose=False,
        )),
    ]),
}



In [18]:
metrics = ["pr_auc", "roc_auc", "recall", "precision", "f1"]
rows = []
fold_results = {}

for model_name, estimator in models.items():
    print(f"=== AVALIANDO MODELO: {model_name} ===")

    cv_res = cross_validate(
        estimator=estimator,
        X=X_train_val,
        y=y_train_val,
        cv=cv,
        scoring=scoring,
        n_jobs=1,
        return_train_score=False,
    )

    fold_results[model_name] = pd.DataFrame({
        "fold": np.arange(1, len(cv_res["fit_time"]) + 1),
        **{metric: cv_res[f"test_{metric}"] for metric in metrics},
        "fit_time_s": cv_res["fit_time"],
        "score_time_s": cv_res["score_time"],
    })

    rows.append({
        "model": model_name,
        **{f"{metric}_mean": cv_res[f"test_{metric}"].mean() for metric in metrics},
        "fit_time_mean_s": cv_res["fit_time"].mean(),
        "score_time_mean_s": cv_res["score_time"].mean(),
    })

results_cv = (
    pd.DataFrame(rows)
    .sort_values("pr_auc_mean", ascending=False)
    .reset_index(drop=True)
)

print("\n=== RESULTADOS MÉDIOS DA VALIDAÇÃO CRUZADA ===")
display(results_cv.round(4))

=== AVALIANDO MODELO: LogisticRegression ===
=== AVALIANDO MODELO: XGBoost ===
=== AVALIANDO MODELO: MLP ===

=== RESULTADOS MÉDIOS DA VALIDAÇÃO CRUZADA ===


,model,pr_auc_mean,roc_auc_mean,recall_mean,precision_mean,f1_mean,fit_time_mean_s,score_time_mean_s
0,LogisticRegression,0.6891,0.8627,0.8218,0.5394,0.6512,0.0563,0.0212
1,MLP,0.6826,0.8591,0.8173,0.5254,0.6390,0.8965,0.0245
2,XGBoost,0.6436,0.8434,0.6605,0.5653,0.6091,0.0715,0.0295


Targeting Encoding

In [19]:
encoding_strategy = "target"

models = {
    "LogisticRegression": Pipeline([
        ("fe", FeatureEngineerTransformer(**round3_fe_params)),
        ("geo", GeoTransformer(
            strategy=encoding_strategy,
            target_smoothing=20.0,
        )),
        ("prep", preprocessor),
        ("scaler", StandardScaler(with_mean=False)),
        ("model", LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=42,
        )),
    ]),
    "XGBoost": Pipeline([
        ("fe", FeatureEngineerTransformer(**round3_fe_params)),
        ("geo", GeoTransformer(
            strategy=encoding_strategy,
            target_smoothing=20.0,
        )),
        ("prep", preprocessor),
        ("model", XGBClassifier(
            objective="binary:logistic",
            eval_metric="logloss",
            scale_pos_weight=(y_train_val == 0).sum() / (y_train_val == 1).sum(),
            random_state=42,
            n_jobs=-1,
        )),
    ]),
    "MLP": Pipeline([
        ("fe", FeatureEngineerTransformer(**round3_fe_params)),
        ("geo", GeoTransformer(
            strategy=encoding_strategy,
            target_smoothing=20.0,
        )),
        ("prep", preprocessor),
        ("scaler", StandardScaler(with_mean=False)),
        ("model", MLPClassifierWrapper(
            hidden_dim=64,
            batch_size=64,
            lr=1e-3,
            weight_decay=1e-5,
            max_epochs=80,
            patience=8,
            val_size=0.15,
            threshold=0.5,
            random_state=42,
            verbose=False,
        )),
    ]),
}



In [20]:
metrics = ["pr_auc", "roc_auc", "recall", "precision", "f1"]
rows = []
fold_results = {}

for model_name, estimator in models.items():
    print(f"=== AVALIANDO MODELO: {model_name} ===")

    cv_res = cross_validate(
        estimator=estimator,
        X=X_train_val,
        y=y_train_val,
        cv=cv,
        scoring=scoring,
        n_jobs=1,
        return_train_score=False,
    )

    fold_results[model_name] = pd.DataFrame({
        "fold": np.arange(1, len(cv_res["fit_time"]) + 1),
        **{metric: cv_res[f"test_{metric}"] for metric in metrics},
        "fit_time_s": cv_res["fit_time"],
        "score_time_s": cv_res["score_time"],
    })

    rows.append({
        "model": model_name,
        **{f"{metric}_mean": cv_res[f"test_{metric}"].mean() for metric in metrics},
        "fit_time_mean_s": cv_res["fit_time"].mean(),
        "score_time_mean_s": cv_res["score_time"].mean(),
    })

results_cv = (
    pd.DataFrame(rows)
    .sort_values("pr_auc_mean", ascending=False)
    .reset_index(drop=True)
)

print("\n=== RESULTADOS MÉDIOS DA VALIDAÇÃO CRUZADA ===")
display(results_cv.round(4))

=== AVALIANDO MODELO: LogisticRegression ===
=== AVALIANDO MODELO: XGBoost ===
=== AVALIANDO MODELO: MLP ===

=== RESULTADOS MÉDIOS DA VALIDAÇÃO CRUZADA ===


,model,pr_auc_mean,roc_auc_mean,recall_mean,precision_mean,f1_mean,fit_time_mean_s,score_time_mean_s
0,MLP,0.6106,0.8248,0.7064,0.5210,0.5996,1.1329,0.0267
1,LogisticRegression,0.6012,0.8199,0.7034,0.5288,0.6034,0.0554,0.0234
2,XGBoost,0.5354,0.7610,0.5053,0.5216,0.5132,0.0740,0.0290


Geo Cluster Encoding

In [21]:
encoding_strategy = "geo_cluster"

models = {
    "LogisticRegression": Pipeline([
        ("fe", FeatureEngineerTransformer(**round3_fe_params)),
        ("geo", GeoTransformer(
            strategy=encoding_strategy,
        )),
        ("prep", preprocessor),
        ("scaler", StandardScaler(with_mean=False)),
        ("model", LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=42,
        )),
    ]),
    "XGBoost": Pipeline([
        ("fe", FeatureEngineerTransformer(**round3_fe_params)),
        ("geo", GeoTransformer(
            strategy=encoding_strategy,
        )),
        ("prep", preprocessor),
        ("model", XGBClassifier(
            objective="binary:logistic",
            eval_metric="logloss",
            scale_pos_weight=(y_train_val == 0).sum() / (y_train_val == 1).sum(),
            random_state=42,
            n_jobs=-1,
        )),
    ]),
    "MLP": Pipeline([
        ("fe", FeatureEngineerTransformer(**round3_fe_params)),
        ("geo", GeoTransformer(
            strategy=encoding_strategy,
        )),
        ("prep", preprocessor),
        ("scaler", StandardScaler(with_mean=False)),
        ("model", MLPClassifierWrapper(
            hidden_dim=64,
            batch_size=64,
            lr=1e-3,
            weight_decay=1e-5,
            max_epochs=80,
            patience=8,
            val_size=0.15,
            threshold=0.5,
            random_state=42,
            verbose=False,
        )),
    ]),
}



In [22]:
metrics = ["pr_auc", "roc_auc", "recall", "precision", "f1"]
rows = []
fold_results = {}

for model_name, estimator in models.items():
    print(f"=== AVALIANDO MODELO: {model_name} ===")

    cv_res = cross_validate(
        estimator=estimator,
        X=X_train_val,
        y=y_train_val,
        cv=cv,
        scoring=scoring,
        n_jobs=1,
        return_train_score=False,
    )

    fold_results[model_name] = pd.DataFrame({
        "fold": np.arange(1, len(cv_res["fit_time"]) + 1),
        **{metric: cv_res[f"test_{metric}"] for metric in metrics},
        "fit_time_s": cv_res["fit_time"],
        "score_time_s": cv_res["score_time"],
    })

    rows.append({
        "model": model_name,
        **{f"{metric}_mean": cv_res[f"test_{metric}"].mean() for metric in metrics},
        "fit_time_mean_s": cv_res["fit_time"].mean(),
        "score_time_mean_s": cv_res["score_time"].mean(),
    })

results_cv = (
    pd.DataFrame(rows)
    .sort_values("pr_auc_mean", ascending=False)
    .reset_index(drop=True)
)

print("\n=== RESULTADOS MÉDIOS DA VALIDAÇÃO CRUZADA ===")
display(results_cv.round(4))

=== AVALIANDO MODELO: LogisticRegression ===
=== AVALIANDO MODELO: XGBoost ===
=== AVALIANDO MODELO: MLP ===

=== RESULTADOS MÉDIOS DA VALIDAÇÃO CRUZADA ===


,model,pr_auc_mean,roc_auc_mean,recall_mean,precision_mean,f1_mean,fit_time_mean_s,score_time_mean_s
0,LogisticRegression,0.6909,0.8625,0.8188,0.5377,0.6489,0.3529,0.0264
1,MLP,0.6873,0.8597,0.8180,0.5206,0.6360,0.8168,0.0331
2,XGBoost,0.6598,0.8481,0.6705,0.5847,0.6245,0.1039,0.0375


ZIP Region Encoding

In [23]:
encoding_strategy = "zip_region"

models = {
    "LogisticRegression": Pipeline([
        ("fe", FeatureEngineerTransformer(**round3_fe_params)),
        ("geo", GeoTransformer(
            strategy=encoding_strategy,
        )),
        ("prep", preprocessor),
        ("scaler", StandardScaler(with_mean=False)),
        ("model", LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=42,
        )),
    ]),
    "XGBoost": Pipeline([
        ("fe", FeatureEngineerTransformer(**round3_fe_params)),
        ("geo", GeoTransformer(
            strategy=encoding_strategy,
        )),
        ("prep", preprocessor),
        ("model", XGBClassifier(
            objective="binary:logistic",
            eval_metric="logloss",
            scale_pos_weight=(y_train_val == 0).sum() / (y_train_val == 1).sum(),
            random_state=42,
            n_jobs=-1,
        )),
    ]),
    "MLP": Pipeline([
        ("fe", FeatureEngineerTransformer(**round3_fe_params)),
        ("geo", GeoTransformer(
            strategy=encoding_strategy,
        )),
        ("prep", preprocessor),
        ("scaler", StandardScaler(with_mean=False)),
        ("model", MLPClassifierWrapper(
            hidden_dim=64,
            batch_size=64,
            lr=1e-3,
            weight_decay=1e-5,
            max_epochs=80,
            patience=8,
            val_size=0.15,
            threshold=0.5,
            random_state=42,
            verbose=False,
        )),
    ]),
}



In [24]:
metrics = ["pr_auc", "roc_auc", "recall", "precision", "f1"]
rows = []
fold_results = {}

for model_name, estimator in models.items():
    print(f"=== AVALIANDO MODELO: {model_name} ===")

    cv_res = cross_validate(
        estimator=estimator,
        X=X_train_val,
        y=y_train_val,
        cv=cv,
        scoring=scoring,
        n_jobs=1,
        return_train_score=False,
    )

    fold_results[model_name] = pd.DataFrame({
        "fold": np.arange(1, len(cv_res["fit_time"]) + 1),
        **{metric: cv_res[f"test_{metric}"] for metric in metrics},
        "fit_time_s": cv_res["fit_time"],
        "score_time_s": cv_res["score_time"],
    })

    rows.append({
        "model": model_name,
        **{f"{metric}_mean": cv_res[f"test_{metric}"].mean() for metric in metrics},
        "fit_time_mean_s": cv_res["fit_time"].mean(),
        "score_time_mean_s": cv_res["score_time"].mean(),
    })

results_cv = (
    pd.DataFrame(rows)
    .sort_values("pr_auc_mean", ascending=False)
    .reset_index(drop=True)
)

print("\n=== RESULTADOS MÉDIOS DA VALIDAÇÃO CRUZADA ===")
display(results_cv.round(4))

=== AVALIANDO MODELO: LogisticRegression ===
=== AVALIANDO MODELO: XGBoost ===
=== AVALIANDO MODELO: MLP ===

=== RESULTADOS MÉDIOS DA VALIDAÇÃO CRUZADA ===


,model,pr_auc_mean,roc_auc_mean,recall_mean,precision_mean,f1_mean,fit_time_mean_s,score_time_mean_s
0,LogisticRegression,0.6902,0.8624,0.8195,0.5376,0.6491,0.0773,0.0241
1,MLP,0.6836,0.8598,0.8341,0.5185,0.6392,0.7293,0.0274
2,XGBoost,0.6470,0.8450,0.6766,0.5735,0.6206,0.1027,0.0326


Risk Band Encoding

In [25]:
encoding_strategy = "risk_band"

models = {
    "LogisticRegression": Pipeline([
        ("fe", FeatureEngineerTransformer(**round3_fe_params)),
        ("geo", GeoTransformer(
            strategy=encoding_strategy,
        )),
        ("prep", preprocessor),
        ("scaler", StandardScaler(with_mean=False)),
        ("model", LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=42,
        )),
    ]),
    "XGBoost": Pipeline([
        ("fe", FeatureEngineerTransformer(**round3_fe_params)),
        ("geo", GeoTransformer(
            strategy=encoding_strategy,
        )),
        ("prep", preprocessor),
        ("model", XGBClassifier(
            objective="binary:logistic",
            eval_metric="logloss",
            scale_pos_weight=(y_train_val == 0).sum() / (y_train_val == 1).sum(),
            random_state=42,
            n_jobs=-1,
        )),
    ]),
    "MLP": Pipeline([
        ("fe", FeatureEngineerTransformer(**round3_fe_params)),
        ("geo", GeoTransformer(
            strategy=encoding_strategy,
        )),
        ("prep", preprocessor),
        ("scaler", StandardScaler(with_mean=False)),
        ("model", MLPClassifierWrapper(
            hidden_dim=64,
            batch_size=64,
            lr=1e-3,
            weight_decay=1e-5,
            max_epochs=80,
            patience=8,
            val_size=0.15,
            threshold=0.5,
            random_state=42,
            verbose=False,
        )),
    ]),
}



In [26]:
metrics = ["pr_auc", "roc_auc", "recall", "precision", "f1"]
rows = []
fold_results = {}

for model_name, estimator in models.items():
    print(f"=== AVALIANDO MODELO: {model_name} ===")

    cv_res = cross_validate(
        estimator=estimator,
        X=X_train_val,
        y=y_train_val,
        cv=cv,
        scoring=scoring,
        n_jobs=1,
        return_train_score=False,
    )

    fold_results[model_name] = pd.DataFrame({
        "fold": np.arange(1, len(cv_res["fit_time"]) + 1),
        **{metric: cv_res[f"test_{metric}"] for metric in metrics},
        "fit_time_s": cv_res["fit_time"],
        "score_time_s": cv_res["score_time"],
    })

    rows.append({
        "model": model_name,
        **{f"{metric}_mean": cv_res[f"test_{metric}"].mean() for metric in metrics},
        "fit_time_mean_s": cv_res["fit_time"].mean(),
        "score_time_mean_s": cv_res["score_time"].mean(),
    })

results_cv = (
    pd.DataFrame(rows)
    .sort_values("pr_auc_mean", ascending=False)
    .reset_index(drop=True)
)

print("\n=== RESULTADOS MÉDIOS DA VALIDAÇÃO CRUZADA ===")
display(results_cv.round(4))

=== AVALIANDO MODELO: LogisticRegression ===
=== AVALIANDO MODELO: XGBoost ===
=== AVALIANDO MODELO: MLP ===

=== RESULTADOS MÉDIOS DA VALIDAÇÃO CRUZADA ===


,model,pr_auc_mean,roc_auc_mean,recall_mean,precision_mean,f1_mean,fit_time_mean_s,score_time_mean_s
0,MLP,0.6124,0.8050,0.6988,0.4904,0.5763,0.8448,0.0256
1,LogisticRegression,0.6122,0.8038,0.6720,0.4990,0.5725,0.0572,0.0235
2,XGBoost,0.5846,0.7919,0.5772,0.5430,0.5593,0.0782,0.0315


Embedding MLP com City

In [27]:
from src.utils.exp import MLPEmbeddingClassifierWrapper

assert "City" in X_train_val.columns, "City precisa estar presente em X_train_val para a rodada de embedding."

embedding_fe_params = dict(round3_fe_params)
embedding_feature_engineer = FeatureEngineerTransformer(**embedding_fe_params)

embedding_models = {
    "MLP_Tabular": Pipeline([
        ("fe", FeatureEngineerTransformer(**embedding_fe_params)),
        ("geo", GeoTransformer(strategy="drop")),
        ("prep", preprocessor),
        ("scaler", StandardScaler(with_mean=False)),
        ("model", MLPClassifierWrapper(
            hidden_dim=64,
            batch_size=64,
            lr=1e-3,
            weight_decay=1e-5,
            max_epochs=80,
            patience=8,
            val_size=0.15,
            threshold=0.5,
            random_state=42,
            verbose=False,
        )),
    ]),
    "MLP_CityEmbedding": MLPEmbeddingClassifierWrapper(
        preprocessor=preprocessor,
        feature_engineer=embedding_feature_engineer,
        city_column="City",
        geo_drop_columns=("Zip Code", "Latitude", "Longitude", "Lat Long"),
        embedding_dim=None,
        hidden_dim=64,
        batch_size=64,
        lr=1e-3,
        weight_decay=1e-5,
        max_epochs=80,
        patience=8,
        val_size=0.15,
        threshold=0.5,
        random_state=42,
        verbose=False,
    ),
}

metrics = ["pr_auc", "roc_auc", "recall", "precision", "f1"]
rows = []
fold_results_embedding = {}

for model_name, estimator in embedding_models.items():
    print(f"=== AVALIANDO MODELO: {model_name} ===")

    cv_res = cross_validate(
        estimator=estimator,
        X=X_train_val,
        y=y_train_val,
        cv=cv,
        scoring=scoring,
        n_jobs=1,
        return_train_score=False,
    )

    fold_results_embedding[model_name] = pd.DataFrame({
        "fold": np.arange(1, len(cv_res["fit_time"]) + 1),
        **{metric: cv_res[f"test_{metric}"] for metric in metrics},
        "fit_time_s": cv_res["fit_time"],
        "score_time_s": cv_res["score_time"],
    })

    rows.append({
        "model": model_name,
        **{f"{metric}_mean": cv_res[f"test_{metric}"].mean() for metric in metrics},
        "fit_time_mean_s": cv_res["fit_time"].mean(),
        "score_time_mean_s": cv_res["score_time"].mean(),
    })

results_embedding = (
    pd.DataFrame(rows)
    .sort_values("pr_auc_mean", ascending=False)
    .reset_index(drop=True)
)

print("\n=== RESULTADOS MLP TABULAR VS EMBEDDING ===")
display(results_embedding.round(4))


=== AVALIANDO MODELO: MLP_Tabular ===
=== AVALIANDO MODELO: MLP_CityEmbedding ===

=== RESULTADOS MLP TABULAR VS EMBEDDING ===


,model,pr_auc_mean,roc_auc_mean,recall_mean,precision_mean,f1_mean,fit_time_mean_s,score_time_mean_s
0,MLP_Tabular,0.6862,0.8604,0.8226,0.5286,0.6431,0.9246,0.0237
1,MLP_CityEmbedding,0.6612,0.8428,0.7668,0.5338,0.6283,0.8234,0.0253


### Conclusão

### Round 4 - Feature Selection

In [16]:
assert "City" in X_train_val.columns, "City precisa estar presente em X_train_val para esta rodada."
assert "Churn Score" in X_train_val.columns, "Churn Score precisa estar presente em X_train_val para esta rodada."
assert "CLTV" not in X_train_val.columns, "CLTV deve permanecer fora de X_train_val como metadata."

selector_label_map = {
    f_classif: "f_classif",
    mutual_info_classif: "mutual_info_classif",
}

round3_fe_params = dict(
    drop_churn_score=False,
    add_engagement_score=True,
    add_tenure_group=True,
    add_tenure_log=True,
    add_contract_ordinal=True,
    add_family_stability=True,
    add_fiber_no_support=True,
)

round3_fe = FeatureEngineerTransformer(**round3_fe_params)
round3_geo = GeoTransformer(strategy="drop")

X_train_val_round3 = round3_fe.fit_transform(X_train_val, y_train_val)
X_train_val_round3 = round3_geo.fit_transform(X_train_val_round3, y_train_val)

processed_feature_names = get_processed_feature_names(
    preprocessor,
    X_train_val_round3,
    y_train_val,
)

k_grid = build_k_grid(
    n_features_processed=len(processed_feature_names),
    min_k=10,
    include_all=True,
)

models = {
    "LogisticRegression": Pipeline([
        ("fe", FeatureEngineerTransformer(**round3_fe_params)),
        ("geo", GeoTransformer(strategy="drop")),
        ("prep", preprocessor),
        ("selector", SelectKBest()),
        ("scaler", StandardScaler(with_mean=False)),
        ("model", LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=42,
        )),
    ]),
    "XGBoost": Pipeline([
        ("fe", FeatureEngineerTransformer(**round3_fe_params)),
        ("geo", GeoTransformer(strategy="drop")),
        ("prep", preprocessor),
        ("selector", SelectKBest()),
        ("model", XGBClassifier(
            objective="binary:logistic",
            eval_metric="logloss",
            scale_pos_weight=(y_train_val == 0).sum() / (y_train_val == 1).sum(),
            random_state=42,
            n_jobs=-1,
        )),
    ]),
    "MLP": Pipeline([
        ("fe", FeatureEngineerTransformer(**round3_fe_params)),
        ("geo", GeoTransformer(strategy="drop")),
        ("prep", preprocessor),
        ("selector", SelectKBest()),
        ("scaler", StandardScaler(with_mean=False)),
        ("model", MLPClassifierWrapper(
            hidden_dim=64,
            batch_size=64,
            lr=1e-3,
            weight_decay=1e-5,
            max_epochs=80,
            patience=8,
            val_size=0.15,
            threshold=0.5,
            random_state=42,
            verbose=False,
        )),
    ]),
}

selector_param_grid = {
    "selector__score_func": [f_classif, mutual_info_classif],
    "selector__k": k_grid,
}

rows = []
feature_selection_searches = {}
selected_feature_logs = {}

for model_name, estimator in models.items():
    print(f"Otimizando seletor de features para {model_name}...")

    search = GridSearchCV(
        estimator=estimator,
        param_grid=selector_param_grid,
        cv=cv,
        scoring=scoring,
        refit="recall",
        return_train_score=False,
        n_jobs=1,
    )
    search.fit(X_train_val, y_train_val)

    feature_selection_searches[model_name] = search
    assert hasattr(search, "best_params_")

    selected_features = extract_selected_feature_names(
        search.best_estimator_,
        processed_feature_names,
        selector_step="selector",
    )
    selected_feature_logs[model_name] = selected_features

    best_k = search.best_params_["selector__k"]
    if best_k == "all":
        assert len(selected_features) == len(processed_feature_names)
    else:
        assert len(selected_features) == best_k

    rows.append(
        summarize_grid_search_results(
            search,
            model_name,
            selector_label_map=selector_label_map,
        )
    )

    print(
        format_selected_features_log(
            model_name,
            search.best_params_,
            selected_features,
        )
    )
    print()

results_fs = (
    pd.DataFrame(rows)
    .sort_values("recall_mean", ascending=False)
    .reset_index(drop=True)
)

assert len(results_fs) == 3, "A rodada de feature selection deve retornar exatamente 3 modelos."

print("=== RESULTADOS FEATURE SELECTION ===")
display(results_fs.round(4))


Otimizando seletor de features para LogisticRegression...
=== FEATURES SELECIONADAS: LogisticRegression ===
Seletor: f_classif
k vencedor: 29
Quantidade final: 29
Features selecionadas:
- cat__Dependents_No
- cat__Dependents_Yes
- cat__Internet Service_Fiber optic
- cat__Internet Service_No
- cat__Online Security_No
- cat__Online Security_No internet service
- cat__Online Backup_No
- cat__Online Backup_No internet service
- cat__Device Protection_No
- cat__Device Protection_No internet service
- cat__Tech Support_No
- cat__Tech Support_No internet service
- cat__Streaming TV_No internet service
- cat__Streaming Movies_No internet service
- cat__Contract_Month-to-month
- cat__Contract_One year
- cat__Contract_Two year
- cat__Paperless Billing_No
- cat__Paperless Billing_Yes
- cat__Payment Method_Electronic check
- num__Tenure Months
- num__Monthly Charges
- num__Total Charges
- num__Churn Score
- num__Tenure_Group_Ordinal
- num__Tenure_Log
- num__Contract_Ordinal
- num__Family_Stability

,model,selector,k,pr_auc_mean,roc_auc_mean,recall_mean,precision_mean,f1_mean,fit_time_mean_s,score_time_mean_s
0,MLP,mutual_info_classif,47,0.9366,0.9749,0.9442,0.7552,0.8389,1.7065,0.0257
1,LogisticRegression,f_classif,29,0.9401,0.9765,0.9373,0.7842,0.8536,0.0471,0.0174
2,XGBoost,f_classif,17,0.9501,0.9801,0.8914,0.8377,0.8637,0.0562,0.0244


L1-Based Selection - Regressão Logística

**Regras**
- Limitar a Regularização para não passar de 10 features (mínimo exigido pelo projeto)

### Conclusão

## Fine Tunning de Hiperparâmetros

**Objetivo:**
- Obter a versão otimizada da MLP e do XGBoost (benchmark);
- Definir EarlyStopping para as otimizações e rodá-las durante 1 hora;
- Avaliar os modelos e comparar com o Baseline (Regressão Logística);
- Fazer o Log dos experimentos no MLFlow

### Grid: MLP

### Grid: XGBoost

### Avaliação

### Conclusão

# Persistindo o Melhor Modelo